### Predicción del precio de autos Toyota Corrolla usados

#### 0 Importar las librerias que necesitamos

In [79]:
#! pip install pandas numpy statsmodels matplotlib seaborn sklearn itertools ml_metrics

In [1]:
#### Importamos las paqueterias necesarias
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from itertools import combinations
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
from ydata_profiling import ProfileReport


#### 1 Cargamos la base de datos y realizamos un reporte

In [2]:
# Cargamos el dataset
car_df = pd.read_csv('ToyotaCorolla.csv')   

In [3]:
# Visualizamos la información del dataset
car_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1436 entries, 0 to 1435
Data columns (total 38 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   model              1436 non-null   object
 1   price              1436 non-null   int64 
 2   age_08_04          1436 non-null   int64 
 3   mfg_month          1436 non-null   int64 
 4   mfg_year           1436 non-null   int64 
 5   km                 1436 non-null   int64 
 6   fuel_type          1436 non-null   object
 7   hp                 1436 non-null   int64 
 8   met_color          1436 non-null   int64 
 9   color              1436 non-null   object
 10  automatic          1436 non-null   int64 
 11  cc                 1436 non-null   int64 
 12  doors              1436 non-null   int64 
 13  cylinders          1436 non-null   int64 
 14  gears              1436 non-null   int64 
 15  quarterly_tax      1436 non-null   int64 
 16  weight             1436 non-null   int64 


In [4]:
# Seleccionamos las variables de interés
car_df = car_df[['price', 'age_08_04', 'km', 'fuel_type', 
                 'hp', 'met_color', 'automatic', 'cc', 'doors', 'quarterly_tax', 'weight']]
car_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1436 entries, 0 to 1435
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   price          1436 non-null   int64 
 1   age_08_04      1436 non-null   int64 
 2   km             1436 non-null   int64 
 3   fuel_type      1436 non-null   object
 4   hp             1436 non-null   int64 
 5   met_color      1436 non-null   int64 
 6   automatic      1436 non-null   int64 
 7   cc             1436 non-null   int64 
 8   doors          1436 non-null   int64 
 9   quarterly_tax  1436 non-null   int64 
 10  weight         1436 non-null   int64 
dtypes: int64(10), object(1)
memory usage: 123.5+ KB


In [5]:
# Realiza un reporte automatizado utilizando ydata-profiling para conocer mejor el dataset. 
profile = ProfileReport(car_df, title="Toyota Corolla Dataset Report")
profile.to_file("toyota_corolla_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 11/11 [00:00<00:00, 65.19it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

#### 2 Limpiamos la base de datos con base en el reporte: duplicados y errores obvios

In [5]:
# Eliminamos los duplicados
car_df = car_df.drop_duplicates()
car_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1435 entries, 0 to 1435
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   price          1435 non-null   int64 
 1   age_08_04      1435 non-null   int64 
 2   km             1435 non-null   int64 
 3   fuel_type      1435 non-null   object
 4   hp             1435 non-null   int64 
 5   met_color      1435 non-null   int64 
 6   automatic      1435 non-null   int64 
 7   cc             1435 non-null   int64 
 8   doors          1435 non-null   int64 
 9   quarterly_tax  1435 non-null   int64 
 10  weight         1435 non-null   int64 
dtypes: int64(10), object(1)
memory usage: 134.5+ KB


In [6]:
duplicados = car_df.duplicated().sum()
print(f"Número de filas duplicadas: {duplicados}")

Número de filas duplicadas: 0


In [7]:
# Transformamos la variable fuel_type en variables dicotómicas con ceros y unos
car_df = pd.get_dummies(
    car_df,
    columns=['fuel_type'],
    drop_first=True,
    dtype=int
)
car_df.head()

,price,age_08_04,km,hp,met_color,automatic,cc,doors,quarterly_tax,weight,fuel_type_Diesel,fuel_type_Petrol
0,13500,23,46986,90,1,0,2000,3,210,1165,1,0
1,13750,23,72937,90,1,0,2000,3,210,1165,1,0
2,13950,24,41711,90,1,0,2000,3,210,1165,1,0
3,14950,26,48000,90,0,0,2000,3,210,1165,1,0
4,13750,30,38500,90,0,0,2000,3,210,1170,1,0


#### 3 Partimos la base de datos en conjunto de entrenamiento y prueba

In [8]:
# Identificamos la variable de interés y las variables predictoras
X = car_df.drop('price', axis=1)
y = car_df['price']
# Hacemos el split de la base de datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
X_train.head()

,age_08_04,km,hp,met_color,automatic,cc,doors,quarterly_tax,weight,fuel_type_Diesel,fuel_type_Petrol
1128,78,109263,110,0,0,1600,5,85,1070,0,1
899,62,59295,86,0,0,1300,5,69,1035,0,1
1188,71,90370,86,1,0,1300,5,69,1035,0,1
311,44,38461,110,1,0,1600,5,85,1080,0,1
1145,75,101855,110,1,0,1600,5,85,1070,0,1


In [10]:
y_train.head()

1128     7500
899      9500
1188     7950
311     13995
1145     6450
Name: price, dtype: int64

#### 4 Limpiamos el conjunto de entranamiento: outliers, missing, escalamiento, etc.

In [11]:
# Aplicamos wenserización para tratar los outliers conjunto de entrenamiento
for col in ['age_08_04', 'km', 'hp', 'cc', 'quarterly_tax', 'weight']:
    X_train[col] = X_train[col].clip(lower=X_train[col].quantile(0.01), upper=X_train[col].quantile(0.99))
# Aplicamos wenserización para tratar los outliers de price
y_train = y_train.clip(lower=y_train.quantile(0.01), upper=y_train.quantile(0.99)) 

/tmp/ipykernel_2027/1257447626.py:3: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train[col] = X_train[col].clip(lower=X_train[col].quantile(0.01), upper=X_train[col].quantile(0.99))


In [12]:
# Duplicados en X_train
duplicados_X_train = X_train.duplicated().sum()
print(f"Número de filas duplicadas en X_train: {duplicados_X_train}")
# Borramos los duplicados en X_train
X_train = X_train.drop_duplicates()
y_train = y_train.loc[X_train.index]  # Aseguramos que y_train se alinee con X_train después de eliminar duplicados
print(f"Número de filas después de eliminar duplicados en X_train: {X_train.shape[0]}")

Número de filas duplicadas en X_train: 2
Número de filas después de eliminar duplicados en X_train: 1146


#### 5 Corremos el modelo de regresión lineal múltiple

In [13]:
# Aplicamos RLM para predecir el precio de los autos
X_train_const = sm.add_constant(X_train)
rlm_model = sm.RLM(y_train, X_train_const, M=sm.robust.norms.HuberT())
rlm_results = rlm_model.fit()
print(rlm_results.summary())

                    Robust linear Model Regression Results                    
Dep. Variable:                  price   No. Observations:                 1146
Model:                            RLM   Df Residuals:                     1134
Method:                          IRLS   Df Model:                           11
Norm:                          HuberT                                         
Scale Est.:                       mad                                         
Cov Type:                          H1                                         
Date:                Tue, 10 Mar 2026                                         
Time:                        20:51:04                                         
No. Iterations:                    10                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const            -2.105e+04   1512.552  

In [14]:
# Evalumos el rendimeinto del modelo con el conjunto de entrenamiento
y_pred_train = rlm_results.predict(sm.add_constant(X_train))
mse_train = mean_squared_error(y_train, y_pred_train)
rmse_train = np.sqrt(mse_train)
mae_train = mean_absolute_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)
print("Métricas de RLM en el conjunto de entrenamiento:")
print(f"RMSE: {rmse_train:.2f}")
print(f"MAE: {mae_train:.2f}")
print(f"R2: {r2_train:.4f}")


Métricas de RLM en el conjunto de entrenamiento:
RMSE: 1220.19
MAE: 906.79
R2: 0.8794


In [15]:
# Precio promedio de los autos en el conjunto de entrenamiento
precio_promedio = y_train.mean()
print(f"Precio promedio de los autos en el conjunto de entrenamiento: {precio_promedio:.2f}")

Precio promedio de los autos en el conjunto de entrenamiento: 10700.67


In [26]:
print(round(mae_train/precio_promedio*100,2),"%")


8.47 %


##### 📈 Interpretación de las métricas del modelo con entrenamiento

##### R² = 0.8794

El modelo explica aproximadamente **88% de la variabilidad** en el precio de los autos.
Esto indica un **muy buen ajuste dentro de muestra**, ya que una proporción alta de la variación del precio está siendo capturada por las variables explicativas.

---

##### MAE = 906.79

En promedio, el modelo se equivoca en aproximadamente **$907 dólares** por auto.
Si el precio promedio es cercano a 10,700, el error relativo es: 8.4% aprox.
Es decir, el modelo comete un error promedio cercano al **9% del precio**, lo cual es bastante razonable en modelos de precios de autos.

---

##### RMSE = 1220.19

El RMSE penaliza más los errores grandes.
El hecho de que: RMSE > MAE, indica que existen algunos errores relativamente grandes (outliers o autos con características difíciles de modelar).
Sin embargo, la diferencia no es extrema, lo que sugiere que el modelo no presenta errores desproporcionadamente altos.

#### 6 Validación cruzada con el modelo del conjunto de entrenamiento

In [27]:
# Aplicamos validación cruzada para evaluar el modelo
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import HuberRegressor
huber_cv_model = HuberRegressor()
cv_scores = cross_val_score(huber_cv_model, X, y, cv=5, scoring='neg_mean_absolute_error')
print(f"MAE Cross-Validation Scores: {-cv_scores}")
print(f"Average MAE: {-cv_scores.mean():.2f}")

MAE Cross-Validation Scores: [2318.09865542 1107.35124977  853.72642263  869.62500671  873.63071365]
Average MAE: 1204.49


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/usr/local/python/3.12.1/lib/python3.12/si

#### 📊 Interpretación de Resultados – Validación Cruzada (MAE)

**Precio promedio del producto:** $10,700  

##### Resultados obtenidos

MAE por fold:
- 2318.06  
- 1105.57  
- 856.35  
- 867.24  
- 877.05  

**MAE promedio:** 1204.85  

---

##### Interpretación del MAE

El **MAE (Mean Absolute Error)** indica que, en promedio, el modelo comete un error de aproximadamente **$1,205 pesos** por predicción.

En términos relativos:

\[
\frac{1204.85}{10700} \approx 11.3\%
\]

Esto significa que el modelo tiene un error promedio cercano al **11% del precio promedio**, lo cual es razonable dependiendo del contexto del mercado y la variabilidad natural de los precios.

---

##### Estabilidad del modelo

Se observa que:

- Cuatro folds presentan errores relativamente similares (entre $856 y $1,105).
- Un fold presenta un error considerablemente mayor ($2,318).

Esto sugiere que el modelo es **generalmente estable**, pero existe cierta sensibilidad a la composición de la muestra en uno de los subconjuntos.

Posibles razones:
- Presencia de outliers en ese fold.
- Segmentos de mercado con comportamiento distinto.
- No linealidad no capturada completamente por el modelo.

---

##### Diagnóstico técnico

✔ El modelo muestra desempeño consistente en la mayoría de los folds.  
⚠ Existe cierta variabilidad que podría indicar:
- Sensibilidad a valores extremos.
- Posible mejora con modelos no lineales.
- Oportunidad de aplicar técnicas robustas o transformaciones.

---

##### Conclusión

El modelo presenta un desempeño aceptable con un error promedio cercano al 11% del precio medio.  
Sin embargo, la variabilidad entre folds sugiere que podría mejorarse mediante:

- Tratamiento de outliers
- Transformaciones (log del precio)
- Modelos más flexibles (árboles o ensambles)

#### 7 Aplicamos los mismos cambios del conjunto de entrenamiento al conjunto prueba: limpiar outliers, etc.

In [28]:
# Aplicamos winserización para tratar los outliers en el conjunto de prueba pero con los parametros del conjunto de entrenamiento
for col in ['age_08_04', 'km', 'hp', 'cc', 'quarterly_tax', 'weight']:
    X_test[col] = X_test[col].clip(lower=X_train[col].quantile(0.01), upper=X_train[col].quantile(0.99))
y_test = y_test.clip(lower=y_train.quantile(0.01), upper=y_train.quantile(0.99))

#### 8 Evaluamos desempeño pero con el conjunto de prueba

In [30]:
# Calculamos las predicciones del modelo utilizando el conjunto de prueba
X_test_const = sm.add_constant(X_test)
y_pred = rlm_results.predict(X_test_const)
# Tabla con los valores reales, las predicciones y los errores
results_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred,
    'Error': y_test - y_pred
})
print(results_df.head().round(2))

      Actual  Predicted    Error
753   9950.0   10476.26  -526.26
858   7995.0   10263.08 -2268.08
630   7500.0    9398.98 -1898.98
1412  9950.0    9302.43   647.57
975   8950.0    8841.10   108.90


In [31]:
# Analizamos las metricas mae, rmse, error medio y r2
mse_lr = mean_squared_error(y_test, y_pred)
rmse_lr = np.sqrt(mse_lr)
mae_lr = mean_absolute_error(y_test, y_pred)
r2_lr = r2_score(y_test, y_pred)
print("Métricas de RLM:")
print(f"RMSE: {rmse_lr:.2f}")
print(f"MAE: {mae_lr:.2f}")
print(f"Error Medio: {np.mean(np.abs(y_test - y_pred)):.2f}")
print(f"R²: {r2_lr:.4f}")

Métricas de RLM:
RMSE: 1256.88
MAE: 929.81
Error Medio: 929.81
R²: 0.8554


In [32]:
# precio promedio de los autos
precio_promedio = y_test.mean()
print(f"Precio promedio de los autos: {precio_promedio:.2f}")

Precio promedio de los autos: 10662.79


#### 9 📈 Interpretación de las métricas del modelo con conjunto prueba

##### R² = 0.88

El modelo explica aproximadamente **88% de la variabilidad** en el precio de los autos.
Esto indica un **muy buen ajuste dentro de muestra**, ya que una proporción alta de la variación del precio está siendo capturada por las variables explicativas.

---

##### MAE = 893.64

En promedio, el modelo se equivoca en aproximadamente **$894 dólares** por auto.
Si el precio promedio es cercano a 10,669, el error relativo es: approx 8.3%.
Es decir, el modelo comete un error promedio cercano al **8%-9% del precio**, lo cual es bastante razonable en modelos de precios de autos.

---

#### RMSE = 1183.8

El RMSE penaliza más los errores grandes.

El hecho de que: RMSE > MAE indica que existen algunos errores relativamente grandes (outliers o autos con características difíciles de modelar).
Sin embargo, la diferencia no es extrema, lo que sugiere que el modelo no presenta errores desproporcionadamente altos.

---

#### Diagnóstico general

✔ El modelo presenta **alta capacidad explicativa (R² alto)**  
✔ El error promedio es **moderado (≈ 8–9%)**  
✔ Existen algunos errores grandes, pero no dominantes  

En conjunto, la Regresión Lineal ofrece un desempeño sólido para predicción de precios, aunque podría mejorar si se capturan posibles **relaciones no lineales** o **interacciones** entre variables.

---

#### Conclusión

El modelo lineal logra explicar gran parte de la variación en los precios y mantiene un error promedio razonable.  
No obstante, dado que los precios suelen presentar comportamientos no lineales (edad, kilometraje, segmento del vehículo), podría explorarse el uso de modelos más flexibles como árboles de regresión para comparar desempeño.